# Real-image encoder performance

Purpose: compare identical full-grid real-image EMA encoder inference using the verified Notebook 00 profiling utilities. Inputs: paired CPU-held models, a fixed BCHW real-image batch and resolved configuration. Outputs: synchronized median/MAD/raw latency, actual attention dispatcher, hardware/precision/workload identity, and separately measured CUDA allocator peak. Expected runtime: seconds to two minutes. Checkpoints: none; source encoders remain unchanged. Resume: rerun `measure_encoder_performance(models,batch,config)` in a fresh kernel after loading the same fixed states and image tensor; each call performs two warmups and five measured repeats. No experiment launches on import.

In [ ]:
from pathlib import Path
import json, os
PROJECT_ROOT = Path(os.environ.get("FMM_JEPA_ROOT", "/home/benjamin/workspace/ahr_go/fmm_jeppa")).expanduser().resolve()
_bootstrap = json.loads((PROJECT_ROOT / "fmm_jepa/bootstrap.ipynb").read_text())
_bootstrap_namespace = {}
for _cell in _bootstrap["cells"]:
    if _cell["cell_type"] == "code" and "ahr-export" in _cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(_cell["source"]), "bootstrap.ipynb#ahr-export", "exec"), _bootstrap_namespace)
_bootstrap_namespace["install_notebook_package"](PROJECT_ROOT)


In [ ]:
"""Matched real-image encoder timing using Notebook 00 synchronization/instrumentation."""
from pathlib import Path
import copy,gc,hashlib,json,os,platform,statistics,time
import torch
from fmm_jepa.evaluation import _sync,_profile_backend
from fmm_jepa.geometry import clear_geometry_cache
from fmm_jepa.tiny_training import isolated_rng


def _encoder_hash(encoder):
    h=hashlib.sha256()
    for key,value in encoder.state_dict().items():
        h.update(key.encode());h.update(str(value.dtype).encode());h.update(str(tuple(value.shape)).encode())
        h.update(value.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()


def measure_encoder_performance(models,batch_or_images,config,warmup=2,repeats=5,checkpoint_ids=None):
    """Return matched complete-grid EMA-encoder forward timing/memory rows, no training."""
    if warmup<2 or repeats<5:raise ValueError('At least two warmup and five measured repeats required')
    if config.get('dtype','float32')!='float32':raise ValueError('Only verified FP32 performance supported')
    images=batch_or_images['images'] if isinstance(batch_or_images,dict) else batch_or_images
    sample_ids=batch_or_images.get('sample_ids',[]) if isinstance(batch_or_images,dict) else []
    if images.ndim!=4 or images.shape[1]!=3:raise ValueError('Expected a BCHW RGB tensor')
    images=images.detach().cpu().float().contiguous()
    image_hash=hashlib.sha256(images.numpy().tobytes()).hexdigest()
    device=torch.device(config['device']);rows=[]
    for model in models.values():
        if any(p.device.type!='cpu' for p in model.parameters()):
            raise ValueError('Keep both source models on CPU to isolate sequential GPU measurements')
    with isolated_rng():
        for kind in ('vit','fmm'):
            clear_geometry_cache();gc.collect()
            if device.type=='cuda':torch.cuda.empty_cache();_sync(device)
            preexisting=torch.cuda.memory_allocated(device) if device.type=='cuda' else None
            original=models[kind].target if hasattr(models[kind],'target') else models[kind]
            original_hash=_encoder_hash(original)
            encoder=copy.deepcopy(original).to(device=device,dtype=torch.float32).eval().requires_grad_(False)
            x=images.to(device)
            patch=encoder.patch_size;grid=[int(x.shape[-2]//patch),int(x.shape[-1]//patch)]
            def call():return encoder(x)
            backend,events=_profile_backend(call,device)
            if kind=='fmm' and events:raise AssertionError('Unexpected attention inside FMM encoder')
            if kind=='vit' and not events:raise AssertionError('Optimized attention dispatcher not observed')
            with torch.no_grad():
                for _ in range(warmup):output=call()
                assert output.shape==(len(x),grid[0]*grid[1],encoder.dim)
                assert bool(torch.isfinite(output).all())
                del output;_sync(device)
                samples=[]
                for _ in range(repeats):
                    _sync(device);start=time.perf_counter();output=call();_sync(device)
                    samples.append(time.perf_counter()-start);del output
                gc.collect();_sync(device)
                if device.type=='cuda':
                    torch.cuda.empty_cache();baseline=torch.cuda.memory_allocated(device)
                    torch.cuda.reset_peak_memory_stats(device);output=call();_sync(device)
                    peak=torch.cuda.max_memory_allocated(device);increment=peak-baseline;del output
                else:baseline=peak=increment=None
            median=statistics.median(samples)
            row={'model_id':kind,'phase':'encoder_forward','split':'train_fixture','status':'PASS',
              'checkpoint_id':(checkpoint_ids or {}).get(kind,original_hash),'encoder_state_sha256':original_hash,
              'image_tensor_sha256':image_hash,'sample_ids':sample_ids,'batch_size':len(images),
              'device':str(device),'device_name':torch.cuda.get_device_name(device) if device.type=='cuda' else platform.processor(),
              'torch_version':str(torch.__version__),'cuda_build':torch.version.cuda,'dtype':'float32','cpu_threads':torch.get_num_threads(),
              'deterministic_algorithms':torch.are_deterministic_algorithms_enabled(),'tf32_matmul':torch.backends.cuda.matmul.allow_tf32,'tf32_cudnn':torch.backends.cudnn.allow_tf32,'cublas_workspace_config':os.environ.get('CUBLAS_WORKSPACE_CONFIG'),
              'grid':grid,'n_grid':grid[0]*grid[1],'n_visible':grid[0]*grid[1],'encoder_dim':encoder.dim,
              'encoder_depth':len(encoder.blocks),'patch_size':patch,'model_config':{**config['model'],'kind':kind},
              'warmup':warmup,'repeats':repeats,'median_seconds':median,
              'mad_seconds':statistics.median(abs(t-median) for t in samples),
              'min_seconds':min(samples),'max_seconds':max(samples),'raw_seconds':samples,
              'images_per_second':len(images)/median,'backend':backend,'backend_events':events,
              'preexisting_cuda_allocated_bytes':preexisting,'baseline_allocated_bytes':baseline,
              'peak_allocated_bytes':peak,'incremental_peak_bytes':increment,
              'memory_definition':'Separate post-warmup max_memory_allocated measurement. Baseline includes this encoder, images, geometric caches and preexisting live allocations; incremental peak measures transient allocations. Excludes reserved allocator and CUDA context memory.' if device.type=='cuda' else 'CPU allocator peak NOT_RUN; process RSS is not substituted.',
              'workload':'EMA target encoder on identical complete real-image grids, final patch LayerNorm, no predictor or backward. No masking or CLS token.',
              'timing_scope':'Warmup excluded; dedicated synchronized wall-clock forward only. Excludes image loading/preprocessing, device transfer, checkpoint loading, feature pooling, classifier, optimizer and EMA.'}
            assert _encoder_hash(original)==original_hash,'Performance measurement mutated source encoder'
            rows.append(row)
            del encoder,x,call;clear_geometry_cache();gc.collect()
            if device.type=='cuda':torch.cuda.empty_cache();_sync(device)
    assert rows[0]['image_tensor_sha256']==rows[1]['image_tensor_sha256']
    assert rows[0]['n_grid']==rows[1]['n_grid'] and rows[0]['batch_size']==rows[1]['batch_size']
    return rows
